----------------------------------------------------------------------------------------------------

# ***FastAPI - Modelos de dados***

----------------------------------------------------------------------------------------------------

# ***1. Alterações necessárias:*** 

## ***1.1 'post.py' na pasta 'controllers':***

No arquivo 'post.py', o seu código será o seguinte:

In [ ]:
from fastapi import Response, status, APIRouter

from schemas.post import PostIn
from views.post import PostOut

router = APIRouter(prefix= "/posts")

@router.post('/', status_code= status.HTTP_201_CREATED, response_model = PostOut)
def create_post(post: PostIn):
    # fake_db.append(post.model_dump())
    return post

@router.get('/', response_model = list[PostOut])
def read_posts(response: Response, published: bool, limit: int, skip: int = 0, ):

    return []

Removemos os códigos que estavão depreciados.

## ***1.2 'post.py' na pasta 'schemas':***

Comparando o arquivo 'post.py' da pasta 'schemas' (Ele é o 'PostIn') com o 'post.py' da pasta 'models', preciamos fazer com que o primeiro possa receber os exatos tipos de posts que definimos no segundo, então no arquivo 'post.py' da pasta 'schemas' ficará assim:

In [ ]:
from datetime import datetime

from pydantic import BaseModel

# Responsável pela detecção da entrada
class PostIn(BaseModel):
    title: str
    content: str
    published_at: datetime | None = None
    published: bool = False

Porque, ao compararmos as duas, o 'post.py' da pasta 'models' esta da seguinte maneira:

In [ ]:
import sqlalchemy
from app.main import metadata

posts = sqlalchemy.Table(
    "posts",
    metadata,
    sqlalchemy.Column("id", sqlalchemy.Integer, primary_key= True),
    sqlalchemy.Column("title", sqlalchemy.String(150), nullable=False),
    sqlalchemy.Column("content", sqlalchemy.String, nullable = False),
    sqlalchemy.Column("published_at", sqlalchemy.DateTime, nullable = True),
    sqlalchemy.Column("published", sqlalchemy.Boolean, default = False)

)

## ***1.3 'post.py' na pasta 'views':***

Assim como foi feito no 'PostIn', devemos fazer no 'PostOut' que esta localizado no arquivo 'post.py', na pasta 'views', ficará assim:

In [ ]:
from datetime import datetime
from pydantic import BaseModel


# responsável pela detecção da saída:
class PostOut(BaseModel):
    title: str
    content: str
    published_at: datetime | None

## ***1.4 'main.py'***

Vamo alterar para da seguinte maneira:

In [ ]:
from contextlib import asynccontextmanager

import databases # importacao necessaria
from fastapi import FastAPI
import sqlalchemy # importacao necessaria
from controllers import post


DATABASE_URL = 'sqlite:///./blog.db'

# cria database
database = databases.Database(DATABASE_URL)
# instanciando a classe SQLalchemmy
metadata = sqlalchemy.MetaData()
engine = sqlalchemy.create_engine(DATABASE_URL, connect_args = {"check_same_thread": False})

@asynccontextmanager
async def lifespan(app: FastAPI):
    from models.post import posts # noqa

    await database.connect()
    # utiliza a engine:
    metadata.create_all(engine)
    yield
    await database.disconnect()

app = FastAPI(lifespan= lifespan)
app.include_router(post.router)
